<a href="https://colab.research.google.com/github/scudilio/AI---Deep-Learning/blob/main/Aula4/MBA_DL_Aula4_02_prompt_llm_sem_treinar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 4 | Case 1: o LLM como classificador e extrator, sem treinar nada

**MBA FIAP | Deep Learning e Redes Neurais Modernas | Profa. Juliana Scudilio**

**Problema de negócio.** O SAC recebe mensagens em texto livre: reviews, e-mails, tickets. Alguém precisa ler, decidir se é reclamação ou elogio, e copiar para o sistema o que o cliente quer. Nas aulas 2 e 3 treinamos modelos para isso (TF-IDF, LSTM, BERT congelado). Hoje a pergunta é: **e se não treinarmos nada?**

**A ideia do método.** Um LLM "instruído" (instruction-tuned) foi treinado para seguir ordens em texto. A ordem é o **prompt**. Trocar o prompt troca a tarefa: o mesmo modelo classifica, extrai campos, resume. Sem dados rotulados, sem treino. O custo aparece em outro lugar: tempo por mensagem e tokens.

**O que vamos medir.** Acurácia do LLM em 200 reviews reais, com e sem exemplos no prompt (zero-shot e few-shot), contra os modelos treinados das aulas anteriores. Depois, extração de campos de um ticket para JSON e a validação do que o modelo inventou.

**Modelo.** `Qwen/Qwen2.5-0.5B-Instruct`: 494 milhões de parâmetros, aberto, roda em CPU. É pequeno de propósito: cabe no Colab gratuito. Modelos maiores (1,5B, 7B, ou uma API) seguem o mesmo código e acertam mais.

**Roteiro (25 min).** 0. Preparar → 1. Primeira conversa → 2. O prompt como programa → 3. Medir em 200 reviews → 4. Few-shot → 5. Extrair campos → 6. O que o modelo inventou → 7. Guardar.

## 0. Preparar o ambiente

A biblioteca `transformers` (Hugging Face) é a porta de entrada para quase todo modelo aberto: ela baixa os pesos do Hugging Face Hub e dá a mesma interface para BERT, GPT, Qwen, Llama. No Colab ela já vem instalada; a linha de `pip` garante uma versão recente.

O arquivo `reviews_b2w_subset.csv` é o mesmo das aulas 2 e 3. Se não estiver na pasta, o Colab abre o seletor de upload.

In [ ]:
%pip install -q "transformers>=4.56" accelerate      # Hugging Face: carregar modelos; accelerate: utilitários de carga

import os                                  # testar se o arquivo existe
import re                                  # expressões regulares: achar "positivo"/"negativo" e o JSON na resposta
import json                                # ler o JSON que o modelo devolve
import time                                # cronometrar
import numpy as np                         # contas com vetores
import pandas as pd                        # tabelas
import torch                               # o motor numérico por trás do modelo

torch.manual_seed(0)                       # reprodutibilidade (o modelo vai gerar de forma determinística, mas é bom hábito)

# achar o CSV na pasta atual ou em data/; se não achar, pedir upload (Colab)
caminho = None
for c in ["reviews_b2w_subset.csv", "../data/reviews_b2w_subset.csv", "data/reviews_b2w_subset.csv"]:
    if os.path.exists(c):
        caminho = c
if caminho is None:
    from google.colab import files
    print("Envie o arquivo reviews_b2w_subset.csv:")
    caminho = list(files.upload())[0]

df = pd.read_csv(caminho)                                  # uma linha por review
df["review_text"] = df["review_text"].astype(str)          # garante texto (há reviews vazios)
df["y"] = (df["sentimento"] == "positivo").astype(int)     # 1 = positivo, 0 = negativo

# a MESMA amostra do notebook 00 desta aula (BERT congelado), para a comparação ser justa
amostra = df.sample(2500, random_state=0)
tr = amostra.iloc[:1500]                   # 1.500 reviews de "treino": aqui só servem para tirar exemplos para o prompt
te = amostra.iloc[1500:]                   # 1.000 reviews de teste, os mesmos do notebook 00
print(df.shape, "| treino", len(tr), "| teste", len(te))

Envie o arquivo reviews_b2w_subset.csv:


## 1. Carregar o LLM e fazer a primeira pergunta

Dois objetos, sempre os mesmos para qualquer LLM da Hugging Face:

- o **tokenizador** (`AutoTokenizer`): transforma texto em números (tokens) e de volta. É o mesmo conceito do notebook 00 (BERT); cada modelo tem o seu;
- o **modelo** (`AutoModelForCausalLM`): "causal LM" é o nome técnico do que o GPT faz: prever o próximo token, um de cada vez, olhando só para o que veio antes.

`dtype=torch.float32` é a precisão dos números na CPU. Em GPU usaríamos `float16` para ocupar metade da memória.

Na primeira execução o Colab baixa o modelo (cerca de 1 GB). Nas seguintes, usa o cache.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

NOME = "Qwen/Qwen2.5-0.5B-Instruct"          # modelo aberto, instruído, 0,5 bilhão de parâmetros

t0 = time.time()
tok = AutoTokenizer.from_pretrained(NOME)                              # baixa o tokenizador
llm = AutoModelForCausalLM.from_pretrained(NOME, dtype=torch.float32)  # baixa os pesos (cerca de 1 GB)
llm.eval()                                                             # modo de inferência (não vamos treinar)

n_param = sum(p.numel() for p in llm.parameters())                     # soma o tamanho de todas as matrizes de pesos
print(f"Carregado em {time.time() - t0:.0f} s | {n_param / 1e6:.0f} milhões de parâmetros | GPU disponível? {torch.cuda.is_available()}")

### Como se fala com um modelo instruído

Um modelo instruído foi treinado com conversas no formato **papel + conteúdo**: `system` (as regras), `user` (o pedido), `assistant` (a resposta). O `apply_chat_template` monta o texto exatamente no formato que o modelo viu no treino, com marcadores especiais. Se mandássemos o texto cru, o modelo responderia pior.

A geração é um laço: o modelo prevê o próximo token, acrescenta à sequência, prevê o próximo... até o limite `max_new_tokens` ou até o token de fim. `do_sample=False` escolhe sempre o token mais provável (determinístico): a mesma pergunta dá sempre a mesma resposta. É o que queremos para classificar; para escrever texto criativo, `do_sample=True` com `temperature` dá variedade.

Esta é a única função do notebook: ela é usada em todas as células seguintes.

In [ ]:
def perguntar(mensagens, max_novos=60, mostrar=True):
    # 1) monta o texto no formato de conversa que o modelo conhece
    texto = tok.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    # 2) texto -> tokens (números); return_tensors="pt" devolve tensores do PyTorch
    entrada = tok(texto, return_tensors="pt")
    n_entrada = entrada["input_ids"].shape[1]                       # quantos tokens entraram
    # 3) gera token a token, sempre o mais provável
    t0 = time.time()
    with torch.no_grad():                                           # não guardar gradientes: só inferência
        saida = llm.generate(**entrada, max_new_tokens=max_novos, do_sample=False, pad_token_id=tok.eos_token_id)
    novos = saida[0][n_entrada:]                                    # só os tokens novos (a resposta)
    resposta = tok.decode(novos, skip_special_tokens=True).strip()  # tokens -> texto
    if mostrar:
        print(f"[{n_entrada} tokens de entrada, {len(novos)} gerados, {time.time() - t0:.1f} s]")
        print(resposta)
    return resposta

r = perguntar([{"role": "user", "content": "Explique em uma frase o que é um modelo de linguagem."}])

### Por dentro: o texto que o modelo recebeu

Olhe os marcadores `<|im_start|>` e `<|im_end|>`: são tokens especiais que separam os papéis. O modelo aprendeu que depois de `<|im_start|>assistant` vem a resposta. Cada modelo tem o seu formato; o `apply_chat_template` esconde isso de nós.

In [ ]:
texto = tok.apply_chat_template([{"role": "user", "content": "Olá!"}], tokenize=False, add_generation_prompt=True)
print(texto)
print("Tokens:", tok(texto)["input_ids"][:12], "...")
print("Em pedaços:", tok.convert_ids_to_tokens(tok(texto)["input_ids"])[:12])

## 2. O prompt como programa: classificar sentimento

Nenhuma linha de treino. A "programação" é o texto do `system`: o que o modelo é, o que deve fazer e **em que formato responder**. Pedir "apenas 'positivo' ou 'negativo'" é o que torna a resposta fácil de usar em código.

Compare com a aula 3: o TF-IDF deu 0% para "Não é ruim" e a LSTM deu 2%. Veja o que o LLM faz com a negação dupla.

In [ ]:
REGRAS = "Você classifica avaliações de produtos de um e-commerce brasileiro. Responda apenas 'positivo' ou 'negativo'."

frases = [
    "Produto excelente, chegou antes do prazo.",
    "Não gostei, veio com defeito e o SAC não resolve.",
    "Chegou rápido, mas parou de funcionar em uma semana.",
    "Não é ruim.",
    "Esperava mais pelo preço.",
]

for frase in frases:
    mensagens = [{"role": "system", "content": REGRAS},                 # as regras
                 {"role": "user", "content": "Avaliação: " + frase}]    # o pedido
    resposta = perguntar(mensagens, max_novos=4, mostrar=False)         # 4 tokens bastam para uma palavra
    print(f"{resposta:<10} | {frase}")

## 3. Medir: 200 reviews reais, zero-shot

"Zero-shot" = o modelo faz a tarefa sem nenhum exemplo dela no prompt. Para medir, precisamos transformar a resposta (texto livre) em 0 ou 1. A regra: se a resposta contém "positivo", é 1; se contém "negativo", é 0; qualquer outra coisa é "indefinido" e conta como erro.

Usamos os 200 primeiros reviews do teste do notebook 00 (os mesmos 1.000 reviews). Cada review leva cerca de meio segundo na CPU; são uns 2 minutos. Em GPU, segundos.

In [ ]:
N = 200
teste = te.iloc[:N]                                            # os 200 primeiros do teste

palpites = []
t0 = time.time()
for texto in teste["review_text"]:
    mensagens = [{"role": "system", "content": REGRAS},
                 {"role": "user", "content": "Avaliação: " + texto[:600]}]   # corta reviews muito longos (tokens custam tempo)
    resposta = perguntar(mensagens, max_novos=4, mostrar=False).lower()
    if "positivo" in resposta:
        palpites.append(1)
    elif "negativo" in resposta:
        palpites.append(0)
    else:
        palpites.append(-1)                                    # resposta fora do formato: conta como erro
tempo_zero = time.time() - t0

palpites = np.array(palpites)
acc_zero = (palpites == teste["y"].values).mean()
print(f"Zero-shot: acurácia {acc_zero:.1%} em {N} reviews | {tempo_zero:.0f} s ({tempo_zero / N * 1000:.0f} ms por review)")
print("Respostas fora do formato:", (palpites == -1).sum())

⏱ **Checkpoint 1:** qual foi a acurácia zero-shot na sua execução? Quantos ms por review?

### Onde o zero-shot erra

Antes de comparar números, leia os erros. Em geral são três tipos: reviews ambíguos, rótulos estranhos (o rótulo veio das estrelas), e casos em que o modelo pequeno "não entendeu" o português coloquial.

In [ ]:
erros = teste[palpites != teste["y"].values]                   # linhas em que o LLM errou
print(f"{len(erros)} erros em {N}\n")
for texto, rotulo in erros[["review_text", "sentimento"]].head(6).values:
    print(f"[rótulo: {rotulo}] {texto[:170]}")

## 4. Few-shot: exemplos dentro do prompt

"Few-shot" = colocar alguns exemplos resolvidos no prompt, como se fossem turnos anteriores da conversa. O modelo não aprende nada (nenhum peso muda); ele **imita o padrão** que vê na conversa. É a forma mais barata de "ensinar" o formato e o critério.

Custo: cada exemplo aumenta o prompt. Mais tokens de entrada = mais tempo e, numa API, mais dinheiro, em **toda** chamada.

Pegamos 4 exemplos do treino (2 positivos, 2 negativos), curtos para não encher o prompt.

In [ ]:
# 4 exemplos curtos do treino: os 2 primeiros positivos e os 2 primeiros negativos com até 120 caracteres
curtos = tr[tr["review_text"].str.len() < 120]
exemplos = pd.concat([curtos[curtos["y"] == 1].head(2), curtos[curtos["y"] == 0].head(2)])

# os exemplos entram como turnos anteriores da conversa: pedido do usuário, resposta do assistente
poucos_tiros = [{"role": "system", "content": REGRAS}]
for texto, rotulo in exemplos[["review_text", "sentimento"]].values:
    poucos_tiros.append({"role": "user", "content": "Avaliação: " + texto})
    poucos_tiros.append({"role": "assistant", "content": rotulo})

for m in poucos_tiros:
    print(f"{m['role']:>9}: {m['content'][:110]}")

In [ ]:
palpites_fs = []
t0 = time.time()
for texto in teste["review_text"]:
    mensagens = poucos_tiros + [{"role": "user", "content": "Avaliação: " + texto[:600]}]   # exemplos + o review novo
    resposta = perguntar(mensagens, max_novos=4, mostrar=False).lower()
    if "positivo" in resposta:
        palpites_fs.append(1)
    elif "negativo" in resposta:
        palpites_fs.append(0)
    else:
        palpites_fs.append(-1)
tempo_fs = time.time() - t0

palpites_fs = np.array(palpites_fs)
acc_fs = (palpites_fs == teste["y"].values).mean()
print(f"Few-shot (4 exemplos): acurácia {acc_fs:.1%} | {tempo_fs:.0f} s ({tempo_fs / N * 1000:.0f} ms por review)")
print("Respostas fora do formato:", (palpites_fs == -1).sum())

### A comparação que importa

Os números de comparação vieram do mesmo arquivo de reviews (notebook 00: mesma amostra de 1.000 reviews de teste; aula 3: 3.000 reviews de teste do mesmo arquivo). O LLM foi medido em 200; a diferença de alguns pontos entre execuções é esperada.

In [ ]:
comp = pd.DataFrame([
    ["TF-IDF + regressão logística (notebook 00, 1.500 reviews)", "1.500 rotulados", "89,1%", "< 1 ms", "sim"],
    ["BERT congelado + regressão logística (notebook 00)",   "1.500 rotulados", "92,9%", "~ 60 ms", "parcial"],
    ["TF-IDF + regressão logística (aula 3, 24 mil reviews)", "24.000 rotulados", "93,9%", "< 1 ms", "sim"],
    ["LSTM com embedding (aula 3)",                           "24.000 rotulados", "92,8%", "~ 0,2 ms", "parcial"],
    [f"LLM 0,5B zero-shot (este notebook, {N} reviews)",      "0 rotulados",      f"{acc_zero:.1%}".replace(".", ","), f"~ {tempo_zero / N * 1000:.0f} ms", "o prompt"],
    [f"LLM 0,5B few-shot, 4 exemplos (este notebook)",        "4 exemplos no prompt", f"{acc_fs:.1%}".replace(".", ","), f"~ {tempo_fs / N * 1000:.0f} ms", "o prompt"],
], columns=["abordagem", "dados necessários", "acurácia", "tempo por review (CPU)", "explicável?"])
comp

**Leitura de negócio.** O LLM pequeno, sem nenhum dado rotulado, chega perto dos modelos treinados com milhares de exemplos. Perde em velocidade por uma ordem de grandeza (centenas de ms contra frações de ms) e não dá um peso por palavra para auditar. Quando vale: **começar** um projeto sem dados rotulados, tarefas que mudam toda semana, volumes pequenos. Quando não vale: milhões de mensagens por dia, tarefa estável, exigência de auditoria. Um modelo maior (ou uma API) sobe a acurácia e o custo.

## 5. Extrair campos de um ticket: texto livre vira JSON

A mesma função, outro prompt, outra tarefa. Agora pedimos um **JSON com campos fixos**. Isso é o que liga o LLM a um sistema: o JSON entra no CRM, abre o chamado, prioriza a fila. A regra "se não estiver no texto, use null" é a defesa contra invenção.

In [ ]:
REGRAS_EXTRACAO = ("Você extrai informações de mensagens de clientes. Responda SOMENTE com um JSON com os campos: "
                   "produto, problema, numero_pedido, pedido_do_cliente, urgencia (baixa, media ou alta). "
                   "Se um campo não estiver no texto, use null.")

tickets = [
    "Comprei uma cafeteira Oster no dia 3 de outubro, pedido 48213. Chegou com o reservatório trincado e vazando. "
    "Já liguei duas vezes no SAC e ninguém resolve. Quero a troca ou o dinheiro de volta até sexta, senão vou ao Procon.",
    "Boa tarde. O fone bluetooth que comprei (pedido 51977) funciona bem, mas a caixa veio amassada. Só queria registrar.",
    "A geladeira chegou e não gela. Comprei há 5 dias. Preciso de assistência urgente, tenho insulina na geladeira.",
]

extraidos = []
for ticket in tickets:
    resposta = perguntar([{"role": "system", "content": REGRAS_EXTRACAO},
                          {"role": "user", "content": ticket}], max_novos=120, mostrar=False)
    trecho = re.search(r"\{.*\}", resposta, re.S)            # acha o primeiro {...} na resposta (o modelo às vezes põe ```json em volta)
    try:
        campos = json.loads(trecho.group())                   # texto -> dicionário Python
    except Exception:
        campos = {"erro": "JSON inválido", "resposta_bruta": resposta}
    extraidos.append(campos)
    print(ticket[:90] + "...")
    print("  ->", campos, "\n")

## 6. O que o modelo inventou: validar antes de usar

Um LLM gera texto plausível, não texto verificado. Para campos extraídos, a validação mais simples e mais útil: **o valor extraído aparece literalmente no ticket?** Se não aparece, alguém precisa olhar. O mesmo vale para o número do pedido, nomes e datas.

A segunda validação é de negócio: a urgência é um julgamento. Compare a urgência que o modelo deu com a que você daria.

In [ ]:
for ticket, campos in zip(tickets, extraidos):
    print(ticket[:70] + "...")
    for campo in ["produto", "numero_pedido"]:
        valor = campos.get(campo)
        if valor is None:
            situacao = "null (modelo não achou)"
        elif str(valor).lower() in ticket.lower():
            situacao = "ok: está no texto"
        else:
            situacao = "ATENÇÃO: não está literalmente no texto, conferir"
        print(f"  {campo:<14} {str(valor):<22} {situacao}")
    print(f"  {'urgencia':<14} {str(campos.get('urgencia')):<22} (julgamento do modelo: você concorda?)\n")

### Resumo: onde a invenção aparece com mais força

Peça um resumo do primeiro ticket e compare, frase a frase, com o original. Modelos pequenos trocam detalhes; modelos grandes também, só com menos frequência. Em produção, resumo de contrato ou de prontuário passa por revisão humana.

In [ ]:
resumo = perguntar([{"role": "user", "content": "Resuma em uma frase, em português, sem acrescentar nada que não esteja no texto: " + tickets[0]}],
                   max_novos=60)
print("\nOriginal:", tickets[0])

## 7. Guardar o resultado

Em projeto, prompt é código: tem versão, teste e métrica. Guardamos o prompt usado, o modelo, a amostra e a acurácia. Se alguém trocar uma vírgula no prompt, a acurácia precisa ser medida de novo.

In [ ]:
resultado = {
    "modelo": NOME,
    "prompt_classificacao": REGRAS,
    "n_teste": N,
    "acc_zero_shot": round(float(acc_zero), 4),
    "acc_few_shot_4_exemplos": round(float(acc_fs), 4),
    "ms_por_review_zero_shot": round(tempo_zero / N * 1000),
    "ms_por_review_few_shot": round(tempo_fs / N * 1000),
}
json.dump(resultado, open("resultado_case1_prompt.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(resultado, indent=2, ensure_ascii=False))

## Para discutir

1. O LLM acertou sem nenhum dado rotulado. Em que situação da sua empresa isso muda o plano do projeto (começar amanhã em vez de esperar 6 meses de rotulagem)?
2. 300 a 500 ms por review na CPU. Com 1 milhão de mensagens por mês, quanto tempo de máquina é isso? E com uma API cobrada por token?
3. O modelo pôs `null` em "pedido_do_cliente" no primeiro ticket, mas o cliente pediu troca ou reembolso. Que tipo de erro é esse e como você o pegaria em produção?

**Próximo case:** o modelo responde sobre o **seu** documento, sem treinar: RAG.